# 💊 Moore Pharmaceuticals — Monte Carlo NPV Simulation

This notebook implements a Monte Carlo simulation of a new drug investment decision for Moore Pharmaceuticals. The goal is to understand the distribution of Net Present Value (NPV) over a five‑year horizon under uncertainty in R&D costs, clinical trial costs, market size, market growth, and market share growth.

The notebook is written for an executive audience:

- Clearly defined assumptions and input distributions
- Transparent calculation of cash flows and NPV
- Summary statistics, percentiles, and confidence intervals
- Plots that make risk and downside visible at a glance


In [ ]:
import numpy as np
import numpy_financial as npf
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True


## 1. Business Context & Assumptions

Moore Pharmaceuticals is evaluating whether to move forward with a new drug. The project requires large upfront investments in R&D and clinical trials, and generates cash flows over a five‑year horizon.

**Baseline financial structure (per case description):**

- R&D costs: total expected around **$700M**
- Clinical trial cost: expected around **$150M**
- Initial market size: **2M patients**, growing annually
- Initial market share: **8%** of the market
- Revenue per monthly prescription: **$130**
- Variable cost per monthly prescription: **$40**
- NPV discount rate: **9%**

Our objective is not just to compute a single NPV, but to understand **how uncertainty in these inputs translates into risk** around the project’s value.

In [ ]:
# --- Core parameters ---
n_years = 5
initial_market_share = 0.08
monthly_revenue_per_patient = 130.0
monthly_cost_per_patient = 40.0
discount_rate = 0.09

# --- Uncertain inputs (distributions) ---
# All values are based on the case assumptions.

# R&D costs: Uniform(600M, 800M)
rd_cost_low = 600_000_000
rd_cost_high = 800_000_000

# Clinical trial costs: Normal(mean=150M, sd=30M)
trial_cost_mean = 150_000_000
trial_cost_std = 30_000_000

# Current market size: Normal(mean=2M patients, sd=0.4M)
market_size_mean = 2_000_000
market_size_std = 400_000

# Annual market growth: Uniform(2%, 6%)
market_growth_low = 0.02
market_growth_high = 0.06

# Annual market share growth: Uniform(15%, 25%)
share_growth_low = 0.15
share_growth_high = 0.25


## 2. NPV Logic for a Single Scenario

For a given draw of the input parameters, we:

1. Start from an initial market size and market share.
2. For each of the next five years:
   - Compute annual profit as *(patients × margin × 12 months)*.
   - Grow market size and market share according to their growth rates.
3. Discount the annual profit stream at 9% to obtain the project NPV **before** upfront costs.
4. Subtract R&D and clinical trial costs to obtain the **project NPV** for that scenario.

In [ ]:
def compute_project_npv(
    current_market_size: float,
    annual_market_growth: float,
    annual_share_growth: float,
    rd_cost: float,
    trial_cost: float,
    n_years: int = n_years,
) -> float:
    """Compute project NPV for a single scenario.

    Parameters
    ----------
    current_market_size : float
        Initial number of potential patients.
    annual_market_growth : float
        Annual growth rate of the total market.
    annual_share_growth : float
        Annual growth rate of Moore's market share.
    rd_cost : float
        Upfront R&D investment.
    trial_cost : float
        Upfront clinical trial cost.
    n_years : int
        Horizon for the NPV calculation.
    """

    market_size = current_market_size
    market_share = initial_market_share

    # Year 0 profit is zero (investment only). We track years 1..n_years.
    annual_profits = np.zeros(n_years + 1)

    margin_per_patient_per_month = monthly_revenue_per_patient - monthly_cost_per_patient

    for year in range(1, n_years + 1):
        patients_on_drug = market_size * market_share
        annual_profits[year] = patients_on_drug * margin_per_patient_per_month * 12

        # Update market size and share for the next year
        market_size *= (1 + annual_market_growth)
        market_share *= (1 + annual_share_growth)

    # Discount the profit stream and subtract upfront costs
    npv_cash_flows = npf.npv(discount_rate, annual_profits)
    return npv_cash_flows - rd_cost - trial_cost


## 3. Monte Carlo Simulation Design

We now simulate many possible futures by repeatedly drawing from the input distributions and computing the NPV each time.

Key design choices:

- Number of iterations: we use 1,000+ scenarios for a stable view of the NPV distribution.
- Random seed: fixed for reproducibility.
- All uncertain inputs are treated as independent draws (for simplicity). This can be relaxed in more advanced versions with correlated variables.


In [ ]:
rng = np.random.default_rng(1)
n_iterations = 1_000

npvs = np.empty(n_iterations)

for i in range(n_iterations):
    # Draw random inputs
    rd_cost = rng.uniform(rd_cost_low, rd_cost_high)
    trial_cost = rng.normal(trial_cost_mean, trial_cost_std)
    current_market_size = rng.normal(market_size_mean, market_size_std)
    annual_market_growth = rng.uniform(market_growth_low, market_growth_high)
    annual_share_growth = rng.uniform(share_growth_low, share_growth_high)

    # Compute NPV for this scenario
    npvs[i] = compute_project_npv(
        current_market_size=current_market_size,
        annual_market_growth=annual_market_growth,
        annual_share_growth=annual_share_growth,
        rd_cost=rd_cost,
        trial_cost=trial_cost,
    )

npvs[:5]  # preview of simulated NPVs


## 4. Summary Statistics & Confidence Interval

For executives, the most important questions are:

- What is the **expected NPV** of the project?
- How wide is the **uncertainty band** around that estimate?
- What is the **probability of a loss** (NPV < 0)?
- How bad can outcomes get in the **left tail**?


In [ ]:
alpha = 0.95  # confidence level

mean_npv = np.mean(npvs)
std_npv = np.std(npvs, ddof=1)

# Normal-approximate confidence interval for the mean NPV
z = norm.ppf((1 + alpha) / 2)
half_width = z * std_npv / np.sqrt(n_iterations)
ci_low = mean_npv - half_width
ci_high = mean_npv + half_width

# Percentiles for the NPV distribution
percentiles = [5, 25, 50, 75, 95]
pct_values = np.percentile(npvs, percentiles)

summary = pd.DataFrame(
    {
        'metric': [
            'mean_npv', 'std_npv', 'ci_low_95pct', 'ci_high_95pct',
        ] + [f'p{p:02d}' for p in percentiles],
        'value': [
            mean_npv,
            std_npv,
            ci_low,
            ci_high,
        ] + list(pct_values),
    }
)

prob_negative = np.mean(npvs < 0)
summary, prob_negative

## 5. Visualizing the NPV Distribution

The histogram below gives a visual sense of the spread of NPV outcomes. For an investment of this scale, understanding the **shape** of the distribution is as important as knowing the mean.

In [ ]:
plt.figure()
plt.hist(npvs, bins=30)
plt.title('Moore Pharmaceuticals — Simulated NPV Distribution')
plt.xlabel('NPV ($)')
plt.ylabel('Frequency')
plt.show()

## 6. Interpretation for Decision Makers

- The **mean NPV** quantifies the central tendency of the project’s value.
- The **95% confidence interval for the mean** shows estimation uncertainty due to finite simulation size.
- The **percentiles** describe the spread, especially the downside (e.g., 5th percentile).
- The **probability of a negative NPV** helps decision makers reason about the risk of capital loss.

In practice, this type of Monte Carlo analysis can be extended to:

- Compare alternative investment strategies
- Stress test key assumptions (e.g., slower market growth, higher costs)
- Explore the impact of correlated uncertainties
